<a href="https://colab.research.google.com/github/bobaboiz0127/bobaboiz0127.github.io/blob/main/GA4_submit_again_ab_test_property_526796832.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITOM6219 Assignment — GA4 Source/Medium A/B Test Notebook

This notebook follows the **Section 2.3 data collection pipeline** style from Week 5.

## Goal
Retrieve **daily GA4 data** for **submit** and **again** performance by **source/medium pair**, print the data, and run **t-tests** to check whether one campaign strictly dominates the others.

## Pipeline
1. Get daily **submit** event counts.
2. Get daily **again** event counts.
3. Get daily **sessions**.
4. Merge the datasets and compute performance metrics.
5. Run pairwise t-tests across source/medium pairs.
6. Check whether one campaign strictly dominates all others.

In [ ]:
# 1. Install packages
!pip3 install google.analytics.data pingouin

In [ ]:
# 2. Imports and credentials
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
    Filter,
    FilterExpression,
)
from pathlib import Path
import os
import pandas as pd
import numpy as np
import pingouin as pg
from itertools import combinations

try:
    from google.colab import userdata
except ImportError:
    userdata = None


def get_credentials_path():
    if userdata is not None:
        try:
            secret_path = userdata.get("GOOGLE_CREDENTIALS_PATH")
            if secret_path:
                return secret_path
        except Exception:
            pass

    candidate_paths = [
        "/mnt/data/bigquery-419120-14d56b3f70eb (1).json",
        "./bigquery-419120-14d56b3f70eb (1).json",
        "./bigquery-419120-14d56b3f70eb.json",
    ]

    for candidate in candidate_paths:
        if os.path.exists(candidate):
            return candidate

    json_matches = list(Path("/mnt/data").glob("*.json")) + list(Path(".").glob("*.json"))
    if json_matches:
        return str(json_matches[0])

    raise FileNotFoundError("Could not find the Google Analytics JSON credentials file.")

In [ ]:
# 3. Configuration
PROPERTY_ID = "526796832"
START_DATE = "2026-01-01"   # change if needed
END_DATE = "today"

SUBMIT_EVENT_NAME = "submit"
AGAIN_EVENT_NAME = "again"

# minimum number of daily observations needed before a campaign enters pairwise tests
MIN_OBS = 3

## 2.3.1 Helper Functions
These functions run GA4 reports and convert API responses into pandas dataframes.

In [ ]:
def response_to_df(response):
    columns = []
    rows = []

    for col in response.dimension_headers:
        columns.append(col.name)
    for col in response.metric_headers:
        columns.append(col.name)

    for row_data in response.rows:
        row = []
        for val in row_data.dimension_values:
            row.append(val.value)
        for val in row_data.metric_values:
            row.append(val.value)
        rows.append(row)

    return pd.DataFrame(rows, columns=columns)


def run_event_report(property_id, event_name, start_date=START_DATE, end_date=END_DATE):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = get_credentials_path()
    client = BetaAnalyticsDataClient()
    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[
            Dimension(name="date"),
            Dimension(name="sessionSource"),  # <-- Updated
            Dimension(name="sessionMedium"),  # <-- Updated
            Dimension(name="eventName"),
        ],
        metrics=[Metric(name="eventCount")],
        date_ranges=[DateRange(start_date=start_date, end_date=end_date)],
        dimension_filter=FilterExpression(
            filter=Filter(
                field_name="eventName",
                string_filter=Filter.StringFilter(value=event_name)
            )
        ),
        limit=100000,
    )
    response = client.run_report(request)
    df = response_to_df(response)

    # Rename columns back so your downstream merge cell still works
    return df.rename(columns={"sessionSource": "source", "sessionMedium": "medium"})


## 2.3.2 Step 1 — Submit Event Data
Retrieve daily submit counts by source/medium.

In [ ]:
submit_df = run_event_report(PROPERTY_ID, SUBMIT_EVENT_NAME)
submit_df = submit_df.rename(columns={"eventCount": "submit_count"})
submit_df["submit_count"] = submit_df["submit_count"].astype(int)

print("Submit event rows:", len(submit_df))
submit_df.head(20)

Submit event rows: 11


,date,source,medium,eventName,submit_count
0,20260314,(direct),(none),submit,30
1,20260316,(direct),(none),submit,14
2,20260315,(direct),(none),submit,8
3,20260314,github.com,referral,submit,4
4,20260319,(direct),(none),submit,2
5,20260324,(direct),(none),submit,2
6,20260311,(direct),(none),submit,1
7,20260312,github.com,referral,submit,1
8,20260316,github.com,referral,submit,1
9,20260319,canvas,announcement,submit,1


## 2.3.3 Step 2 — Again Event Data
Retrieve daily again counts by source/medium.

In [ ]:
again_df = run_event_report(PROPERTY_ID, AGAIN_EVENT_NAME)
again_df = again_df.rename(columns={"eventCount": "again_count"})
again_df["again_count"] = again_df["again_count"].astype(int)

print("Again event rows:", len(again_df))
again_df.head(20)

Again event rows: 26


,date,source,medium,eventName,again_count
0,20260324,campaignlaunch,banner,again,18
1,20260314,(direct),(none),again,17
2,20260321,banner,Lara,again,11
3,20260315,(direct),(none),again,10
4,20260317,(direct),(none),again,9
5,20260317,github.com,referral,again,9
6,20260320,Banner,Eduardo,again,5
7,20260318,(direct),(none),again,4
8,20260319,github.com,referral,again,4
9,20260322,banner,Lara,again,4


## 2.3.4 Step 3 — Session Data
Retrieve daily session totals by source/medium.

In [ ]:
sessions_df = run_sessions_report(PROPERTY_ID)
sessions_df["sessions"] = sessions_df["sessions"].astype(int)

print("Session rows:", len(sessions_df))
sessions_df.head(20)

Session rows: 46


,date,source,medium,sessions
0,20260326,(direct),(none),8
1,20260322,banner,Lara,6
2,20260321,banner,Lara,5
3,20260324,banner,Quoc,5
4,20260326,banner,Quoc,4
5,20260323,(direct),(none),3
6,20260324,(direct),(none),3
7,20260324,Banner,Eduardo,3
8,20260325,banner,Lara,3
9,20260321,github.com,referral,2


## 2.3.5 Step 4 — Merge the Datasets and Build Daily Performance Measures

We compute:
- `submit_rate` = submit_count / sessions
- `again_rate` = again_count / sessions
- `submit_share` = submit_count / (submit_count + again_count)

`submit_share` is the main comparison metric here because it summarizes submit-versus-again performance on a 0 to 1 scale.

In [ ]:
submit_small = submit_df[["date", "source", "medium", "submit_count"]].copy()
again_small = again_df[["date", "source", "medium", "again_count"]].copy()
sessions_small = sessions_df[["date", "source", "medium", "sessions"]].copy()

merged_df = sessions_small.merge(
    submit_small, on=["date", "source", "medium"], how="left"
).merge(
    again_small, on=["date", "source", "medium"], how="left"
)

merged_df["submit_count"] = merged_df["submit_count"].fillna(0).astype(int)
merged_df["again_count"] = merged_df["again_count"].fillna(0).astype(int)
merged_df["sessions"] = merged_df["sessions"].fillna(0).astype(int)

merged_df["submit_rate"] = np.where(
    merged_df["sessions"] > 0,
    merged_df["submit_count"] / merged_df["sessions"],
    np.nan
)

merged_df["again_rate"] = np.where(
    merged_df["sessions"] > 0,
    merged_df["again_count"] / merged_df["sessions"],
    np.nan
)

merged_df["submit_share"] = np.where(
    (merged_df["submit_count"] + merged_df["again_count"]) > 0,
    merged_df["submit_count"] / (merged_df["submit_count"] + merged_df["again_count"]),
    np.nan
)

merged_df["campaign"] = merged_df["source"] + " / " + merged_df["medium"]

merged_df = merged_df.sort_values(["campaign", "date"]).reset_index(drop=True)

print("Merged daily dataset:")
print(merged_df.shape)
merged_df.head(50)

Merged daily dataset:
(46, 10)


,date,source,medium,sessions,submit_count,again_count,submit_rate,again_rate,submit_share,campaign
0,20260319,(direct),(none),1,2,3,2.000000,3.000000,0.4,(direct) / (none)
1,20260323,(direct),(none),3,0,0,0.000000,0.000000,NaN,(direct) / (none)
2,20260324,(direct),(none),3,2,2,0.666667,0.666667,0.5,(direct) / (none)
3,20260325,(direct),(none),1,0,3,0.000000,3.000000,0.0,(direct) / (none)
4,20260326,(direct),(none),8,0,2,0.000000,0.250000,0.0,(direct) / (none)
5,20260320,Banner,Eduardo,1,0,5,0.000000,5.000000,0.0,Banner / Eduardo
6,20260322,Banner,Eduardo,1,0,0,0.000000,0.000000,NaN,Banner / Eduardo
7,20260323,Banner,Eduardo,1,0,0,0.000000,0.000000,NaN,Banner / Eduardo
8,20260324,Banner,Eduardo,3,0,2,0.000000,0.666667,0.0,Banner / Eduardo
9,20260325,Banner,Eduardo,2,0,0,0.000000,0.000000,NaN,Banner / Eduardo


## Print the Daily Data
This is the table you can show as evidence of the GA4 retrieval step.

In [ ]:
print(
    merged_df[
        ["date", "source", "medium", "sessions", "submit_count", "again_count", "submit_rate", "again_rate", "submit_share"]
    ].to_string(index=False)
)

    date                            source       medium  sessions  submit_count  again_count  submit_rate  again_rate  submit_share
20260319                          (direct)       (none)         1             2            3     2.000000    3.000000           0.4
20260323                          (direct)       (none)         3             0            0     0.000000    0.000000           NaN
20260324                          (direct)       (none)         3             2            2     0.666667    0.666667           0.5
20260325                          (direct)       (none)         1             0            3     0.000000    3.000000           0.0
20260326                          (direct)       (none)         8             0            2     0.000000    0.250000           0.0
20260320                            Banner      Eduardo         1             0            5     0.000000    5.000000           0.0
20260322                            Banner      Eduardo         1           

## 2.3.6 Step 5 — Keep Campaigns With Enough Daily Observations
This avoids running unstable t-tests on campaigns with too little data.

In [ ]:
campaign_counts = (
    merged_df.groupby("campaign")["submit_share"]
    .apply(lambda s: s.notna().sum())
    .reset_index(name="n_days_with_submit_or_again")
    .sort_values("n_days_with_submit_or_again", ascending=False)
)

eligible_campaigns = campaign_counts.loc[
    campaign_counts["n_days_with_submit_or_again"] >= MIN_OBS, "campaign"
].tolist()

print("Campaign observation counts:")
display(campaign_counts)

print("\nEligible campaigns for pairwise tests:")
print(eligible_campaigns)

Campaign observation counts:


,campaign,n_days_with_submit_or_again
0,(direct) / (none),4
7,banner / Lara,3
8,banner / Quoc,3
1,Banner / Eduardo,2
10,github.com / referral,2
12,usc-word-edit.officeapps.live.com / referral,1
9,campaignlaunch / banner,1
4,Email Newslettere / Email,1
2,Canvas / announcement,0
3,Email Newsletter / Email,0



Eligible campaigns for pairwise tests:
['(direct) / (none)', 'banner / Lara', 'banner / Quoc']


## 2.3.7 Pairwise T-Tests Across Source/Medium Pairs

We compare campaigns using **daily `submit_share`**.
- A higher mean means a larger share of the action ended in **submit** instead of **again**.
- We report both **Student's t-test** and **Welch's t-test**.
- The main conclusion should rely more on **Welch's t-test** when variances or sample sizes differ.

In [ ]:
pairwise_results = []

for campaign_a, campaign_b in combinations(eligible_campaigns, 2):
    a = merged_df.loc[merged_df["campaign"] == campaign_a, "submit_share"].dropna()
    b = merged_df.loc[merged_df["campaign"] == campaign_b, "submit_share"].dropna()

    if len(a) < MIN_OBS or len(b) < MIN_OBS:
        continue

    student = pg.ttest(a, b, correction=False)
    welch = pg.ttest(a, b, correction=True)

    mean_a = a.mean()
    mean_b = b.mean()

    pairwise_results.append({
        "campaign_a": campaign_a,
        "campaign_b": campaign_b,
        "n_a": len(a),
        "n_b": len(b),
        "mean_submit_share_a": mean_a,
        "mean_submit_share_b": mean_b,
        "student_p": student["p_val"].iloc[0],
        "welch_p": welch["p_val"].iloc[0],
        "student_ci95": str(student["CI95"].iloc[0]),
        "welch_ci95": str(welch["CI95"].iloc[0]),
        "better_mean_campaign": campaign_a if mean_a > mean_b else campaign_b,
        "welch_significant_5pct": bool(welch["p_val"].iloc[0] < 0.05),
    })

pairwise_df = pd.DataFrame(pairwise_results)

if not pairwise_df.empty:
    pairwise_df = pairwise_df.sort_values(
        ["welch_significant_5pct", "welch_p"], ascending=[False, True]
    )
else:
    print("Not enough data to run pairwise tests.")

pairwise_df

/usr/local/lib/python3.12/dist-packages/pingouin/effsize.py:831: RuntimeWarning: invalid value encountered in scalar divide
  d = (x.mean() - y.mean()) / poolsd
/usr/local/lib/python3.12/dist-packages/pingouin/effsize.py:831: RuntimeWarning: invalid value encountered in scalar divide
  d = (x.mean() - y.mean()) / poolsd


,campaign_a,campaign_b,n_a,n_b,mean_submit_share_a,mean_submit_share_b,student_p,welch_p,student_ci95,welch_ci95,better_mean_campaign,welch_significant_5pct
0,(direct) / (none),banner / Lara,4,3,0.225,0.0,0.207782,0.185596,[-0.17 0.62],[-0.19 0.64],(direct) / (none),False
1,(direct) / (none),banner / Quoc,4,3,0.225,0.0,0.207782,0.185596,[-0.17 0.62],[-0.19 0.64],(direct) / (none),False
2,banner / Lara,banner / Quoc,3,3,0.000,0.0,NaN,NaN,[nan nan],[nan nan],banner / Quoc,False


## 2.3.8 Campaign Summary Table
This table shows the average daily performance before hypothesis testing conclusions.

In [ ]:
campaign_summary = (
    merged_df.groupby("campaign")
    .agg(
        days=("date", "nunique"),
        avg_sessions=("sessions", "mean"),
        avg_submit_count=("submit_count", "mean"),
        avg_again_count=("again_count", "mean"),
        avg_submit_rate=("submit_rate", "mean"),
        avg_again_rate=("again_rate", "mean"),
        avg_submit_share=("submit_share", "mean"),
    )
    .reset_index()
    .sort_values("avg_submit_share", ascending=False)
)

campaign_summary

,campaign,days,avg_sessions,avg_submit_count,avg_again_count,avg_submit_rate,avg_again_rate,avg_submit_share
0,(direct) / (none),5,3.200000,0.8,2.000000,0.533333,1.383333,0.225
1,Banner / Eduardo,6,1.666667,0.0,1.166667,0.000000,0.944444,0.000
4,Email Newslettere / Email,2,1.500000,0.0,1.000000,0.000000,0.500000,0.000
7,banner / Lara,6,2.833333,0.0,2.666667,0.000000,0.533333,0.000
8,banner / Quoc,6,2.500000,0.0,1.166667,0.000000,0.650000,0.000
9,campaignlaunch / banner,3,1.000000,0.0,6.000000,0.000000,6.000000,0.000
10,github.com / referral,5,1.400000,0.0,1.000000,0.000000,0.900000,0.000
12,usc-word-edit.officeapps.live.com / referral,1,1.000000,0.0,1.000000,0.000000,1.000000,0.000
2,Canvas / announcement,2,1.000000,0.0,0.000000,0.000000,0.000000,NaN
3,Email Newsletter / Email,2,1.500000,0.0,0.000000,0.000000,0.000000,NaN


## 2.3.9 Strict Dominance Check

A campaign **strictly dominates** others here if:
1. It has the highest mean `submit_share`, and
2. In Welch pairwise tests, it is significantly better than every other eligible campaign at the 5% level.

In [ ]:
if campaign_summary.empty or pairwise_df.empty:
    print("Not enough eligible campaigns to run a strict dominance check.")
else:
    top_campaign = campaign_summary.iloc[0]["campaign"]

    top_vs_others = pairwise_df[
        (pairwise_df["campaign_a"] == top_campaign) |
        (pairwise_df["campaign_b"] == top_campaign)
    ].copy()

    def top_campaign_wins(row):
        if row["campaign_a"] == top_campaign:
            return row["mean_submit_share_a"] > row["mean_submit_share_b"]
        return row["mean_submit_share_b"] > row["mean_submit_share_a"]

    top_vs_others["top_campaign_has_higher_mean"] = top_vs_others.apply(top_campaign_wins, axis=1)

    strictly_dominates = (
        len(top_vs_others) == len(eligible_campaigns) - 1 and
        top_vs_others["welch_significant_5pct"].all() and
        top_vs_others["top_campaign_has_higher_mean"].all()
    )

    print("Top campaign by mean submit_share:", top_campaign)
    display(top_vs_others)

    if strictly_dominates:
        print(f"Conclusion: {top_campaign} strictly dominates all other eligible campaigns based on Welch t-tests.")
    else:
        print(f"Conclusion: {top_campaign} does NOT strictly dominate all other eligible campaigns.")

Top campaign by mean submit_share: (direct) / (none)


,campaign_a,campaign_b,n_a,n_b,mean_submit_share_a,mean_submit_share_b,student_p,welch_p,student_ci95,welch_ci95,better_mean_campaign,welch_significant_5pct,top_campaign_has_higher_mean
0,(direct) / (none),banner / Lara,4,3,0.225,0.0,0.207782,0.185596,[-0.17 0.62],[-0.19 0.64],(direct) / (none),False,True
1,(direct) / (none),banner / Quoc,4,3,0.225,0.0,0.207782,0.185596,[-0.17 0.62],[-0.19 0.64],(direct) / (none),False,True


Conclusion: (direct) / (none) does NOT strictly dominate all other eligible campaigns.


## Final Written Conclusion Template

Use the printed output to write your final conclusion.

Example structure:

- I retrieved daily GA4 data for **submit**, **again**, and **sessions** by **source/medium pair**.
- I merged the reports and computed **submit_share = submit / (submit + again)** as the main performance metric.
- I ran pairwise **Student's** and **Welch's** t-tests across eligible campaigns.
- I used **Welch's t-test** as the main decision rule when variances or sample sizes differed.
- Based on the results, **[top campaign]** [does / does not] strictly dominate the other campaigns.

Replace the bracketed part after you run the notebook.